In [1]:
from scipy.stats import spearmanr, ttest_ind, f_oneway
import pandas as pd
from config.config import DATASET_DIR

In [2]:
def analyze_numeric(df, target):
    results = []

    for col in df.select_dtypes(include=['number']).columns:
        if col == target:
            continue

        temp = df[[col, target]].dropna()

        if len(temp) < 30:
            continue

        corr, p = spearmanr(temp[col], temp[target])

        results.append({
            "feature": col,
            "correlation": corr,
            "p_value": p
        })

    return sorted(results, key=lambda x: x["p_value"])

In [3]:
def analyze_categorical(df, target):
    results = []

    for col in df.select_dtypes(include=['object', 'category', 'bool']).columns:

        temp = df[[col, target]].dropna()
        groups = temp.groupby(col)[target].apply(list)

        if len(groups) < 2:
            continue

        if len(groups) == 2:
            stat, p = ttest_ind(groups.iloc[0], groups.iloc[1], equal_var=False)
        else:
            stat, p = f_oneway(*groups)

        results.append({
            "feature": col,
            "p_value": p,
            "groups": len(groups)
        })

    return sorted(results, key=lambda x: x["p_value"])

In [4]:
df = pd.read_csv(f"{DATASET_DIR}/final.csv")

In [5]:
target = "LOS"  

df[target] = pd.to_numeric(df[target], errors="coerce")

numeric_results = pd.DataFrame(analyze_numeric(df, target))
categorical_results = pd.DataFrame(analyze_categorical(df, target))

print("Numeric features associated with LOS:")
display(numeric_results)

print("Categorical features associated with LOS:")
display(categorical_results)

C:\Users\belac\AppData\Local\Temp\ipykernel_5840\1289673303.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object', 'category', 'bool']).columns:


Numeric features associated with LOS:


,feature,correlation,p_value
0,gcs_eye_min,-0.274216,0.000000
1,gcs_motor_min,-0.255420,0.000000
2,gcs_total_min,-0.278951,0.000000
3,gcs_verbal_min,-0.297559,0.000000
4,respiratory_rate_min,-0.191583,0.000000
...,...,...,...
201,fluconazole_sum,-0.020927,0.752775
202,ketamine_count,0.033760,0.803125
203,dialysis_out_count,-0.038445,0.806641
204,d10w_sum,-0.012771,0.849583


Categorical features associated with LOS:


,feature,p_value,groups
0,DOB,0.000000e+00,32517
1,ADMISSION_TYPE,0.000000e+00,4
2,ADMITTIME,3.537865e-231,57490
3,INTIME,3.120745e-02,61521
4,END_FIRST_24H,3.120745e-02,61521
5,GENDER,4.293002e-01,2
6,DIAGNOSIS,1.000000e+00,15248


In [6]:
alpha = 0.01

significant_numeric = numeric_results[numeric_results["p_value"] < alpha]
significant_categorical = categorical_results[categorical_results["p_value"] < alpha]

print("Significant numeric features:")
display(significant_numeric)

print("Significant categorical features:")
display(significant_categorical)

Significant numeric features:


,feature,correlation,p_value
0,gcs_eye_min,-0.274216,0.000000
1,gcs_motor_min,-0.255420,0.000000
2,gcs_total_min,-0.278951,0.000000
3,gcs_verbal_min,-0.297559,0.000000
4,respiratory_rate_min,-0.191583,0.000000
...,...,...,...
138,albumin_5_count,0.065977,0.007201
139,piperacillin_tazo_sum,0.052023,0.007217
140,cisatracurium_sum,0.158360,0.007289
141,cryoprecipitate_sum,0.144482,0.007714


Significant categorical features:


,feature,p_value,groups
0,DOB,0.000000e+00,32517
1,ADMISSION_TYPE,0.000000e+00,4
2,ADMITTIME,3.537865e-231,57490
